In [1]:
from pathlib import Path
import os

In [2]:
from sourceContentProcessor.extractor import PDFExtractor
# Import ContentParser directly first (fast, no dependencies)
from sourceContentProcessor.parser.content_parser import ContentParser

# NOTE: TOCParser import is slow (imports langchain_core) - uncomment only when needed
from sourceContentProcessor.parser.toc_parser import TOCParser


In [3]:
BASE_DIR_PATH = Path.cwd().parent.parent
TEXTBOOK_PATH = BASE_DIR_PATH / "docs" / "textbooks"
print(TEXTBOOK_PATH)

c:\Users\Motunrayo Ibiyo\Documents\IBrary\docs\textbooks


In [4]:
biology_textbook_path = TEXTBOOK_PATH / "biology"
essential_biology_path = biology_textbook_path / "essential-biology.pdf"

In [5]:
os.path.exists(essential_biology_path)

True

In [6]:
pdfExtractor = PDFExtractor()

In [7]:
content_parser = ContentParser()

In [8]:
essential_biology_docs = pdfExtractor.extract(essential_biology_path)

In [9]:
essential_biology_docs[19]

Document(metadata={'producer': 'GPL Ghostscript 9.14', 'creator': 'PDF24 Creator', 'creationdate': '2017-03-28T05:45:45+03:00', 'moddate': '2017-03-28T05:45:45+03:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'source': 'c:\\Users\\Motunrayo Ibiyo\\Documents\\IBrary\\docs\\textbooks\\biology\\essential-biology.pdf', 'total_pages': 718, 'page': 19, 'page_label': '1'}, page_content='1\n1\nOUTLINE\n 1.1  The Characteristics of Life 2\n 1.2 Evolution: The Core Concept of \nBiology 6\n 1.3 Science: A Way of Knowing 1 1\n 1.4 Challenges Facing Science 16\nThe Diversity of Life\nLife on Earth takes on a staggering variety of forms, often with appearances \nand behaviors that may be strange to humans. As we will see in this chapter, \none of the ways that biologists classify life is by species. So how many specie s \nare there on the planet? The truth is, we really don’t know. Recent estimates \nsuggest that there may be around 8.7 million species on the planet, but many \nscie

In [10]:
# Parse Table of Contents first
# Parse detailed TOC (indices 11-18) - includes subtopics and subsubtopics

toc_parser = TOCParser()

In [11]:
# Parse brief TOC (index 3) - chapter numbers, titles, and page numbers
brief_toc = toc_parser.parse_brief_toc(essential_biology_docs[3])
print("Number of chapters:", len(brief_toc))
print("Brief TOC (first 5 chapters):")
for ch in brief_toc[:5]:
    print(f"  Chapter {ch['chapter_number']}: {ch['chapter_title']} (page_label: {ch['page_label']})")

# Parse detailed TOC (indices 11-18) - includes subtopics and subsubtopics
detailed_toc = toc_parser.parse_detailed_toc(
    essential_biology_docs[11:19],  # 11-18 inclusive
    brief_toc=brief_toc  # Required: Maps subtopics to chapters by prefix
)
print(f"\nDetailed TOC parsed: {len(detailed_toc)} chapters found")

# Create page_label to index mapping
page_label_map = toc_parser.create_page_label_to_index_map(essential_biology_docs)
print(f"\nPage label map created: {len(page_label_map)} page labels mapped")
print(f"Example: page_label '1' -> index {page_label_map.get('1', 'Not found')}")

Number of chapters: 32
Brief TOC (first 5 chapters):
  Chapter 1: Biology: The Science of Life (page_label: 1)
  Chapter 2: The Chemical Basis of Life (page_label: 21)
  Chapter 3: The Organic Molecules of Life (page_label: 38)
  Chapter 4: Inside the Cell (page_label: 56)
  Chapter 5: The Dynamic Cell (page_label: 79)

Detailed TOC parsed: 32 chapters found

Page label map created: 718 page labels mapped
Example: page_label '1' -> index 19


In [12]:
def create_outline(subtopics: list[dict[str, str]]) -> str:
    outline = ''
    for subtopic in subtopics:
        outline += f"{subtopic['subtopic_number']}. {subtopic['subtopic_title']}\n"
    return outline

In [21]:
def extract_subtopic_info(
    subtopic: dict[str, str],
    next_subtopic: dict[str, str] | None,
    next_chapter: dict[str, str],
) -> dict[str, str]:
    subtopic_info = {}
    subtopic_number = subtopic['subtopic_number']
    subtopic_title = subtopic['subtopic_title']
    subtopic_page_label = subtopic['page_label']
    subsubtopics = subtopic.get('subsubtopics', [])

    subtopic_page_doc_index = page_label_map[subtopic_page_label]
    page_content = essential_biology_docs[subtopic_page_doc_index].page_content

    # Defaults so we never reference an unassigned value
    subtopic_info['learning_outcomes'] = ''
    subtopic_info['subtopic_introduction'] = ''

    if subsubtopics:
        first_subsubtopic_title = subsubtopics[0]['subsubtopic_title']
        subtopic_page_contents = content_parser.parse_subtopic_content(
            page_content,
            subtopic_title,
            first_subsubtopic_title,
        )
        subtopic_info['learning_outcomes'] = subtopic_page_contents.get('learning_outcomes', '')
        subtopic_info['subtopic_introduction'] = subtopic_page_contents.get('subtopic_introduction', '')
    else:
        # No subsubtopics: capture content until the next subtopic (or next chapter).
        if next_subtopic:
            next_title = next_subtopic['subtopic_title']
            next_page_label = next_subtopic['page_label']
        else:
            next_title = "ASSESS"
            next_page_label = next_chapter['page_label']

        next_page_doc_index = page_label_map[next_page_label] - 1
        documents_for_subtopic = essential_biology_docs[
            subtopic_page_doc_index:next_page_doc_index + 1
        ]
        page_content_for_subtopic = [doc.page_content for doc in documents_for_subtopic]
        subtopic_content = content_parser.parse_subsubtopic_content(
            page_content_for_subtopic,
            subtopic_title,
            next_title,
        )

        subtopic_info['subtopic_start_page_label'] = subtopic_page_label
        subtopic_info['subtopic_end_page_label'] = next_page_label
        subtopic_info['subtopic_content'] = subtopic_content

    subtopic_info['subtopic_number'] = subtopic_number
    subtopic_info['subtopic_title'] = subtopic_title
    subtopic_info['subtopic_page_label'] = subtopic_page_label

    return subtopic_info

In [19]:

def extract_subsubtopic_info(
    current_subsubtopic: dict[str, str],
    next_subsubtopic: dict[str, str] | None,
    next_chapter: dict[str, str] | None,
) -> dict[str, str]:
    subsubtopic_info = {}
    current_subsubtopic_title = current_subsubtopic['subsubtopic_title']
    current_subsubtopic_page_label = current_subsubtopic['page_label']
    current_subsubtopic_page_doc_index = page_label_map[current_subsubtopic_page_label]

    # Default to next chapter boundary; override if a next subsubtopic exists.
    next_subsubtopic_title = "ASSESS"
    next_subsubtopic_page_label = None
    if next_chapter:
        next_subsubtopic_page_label = next_chapter['page_label']

    if next_subsubtopic:
        next_subsubtopic_title = next_subsubtopic['subsubtopic_title']
        next_subsubtopic_page_label = next_subsubtopic['page_label']

    if next_subsubtopic_page_label in page_label_map:
        next_subsubtopic_page_doc_index = page_label_map[next_subsubtopic_page_label] - 1
    else:
        next_subsubtopic_page_doc_index = current_subsubtopic_page_doc_index

    if next_subsubtopic_page_doc_index < current_subsubtopic_page_doc_index:
        next_subsubtopic_page_doc_index = current_subsubtopic_page_doc_index

    documents_for_subsubtopic_doc = essential_biology_docs[
        current_subsubtopic_page_doc_index:next_subsubtopic_page_doc_index + 1
    ]

    page_content_for_subsubtopic = [doc.page_content for doc in documents_for_subsubtopic_doc]
    subsubtopic_content = content_parser.parse_subsubtopic_content(
        page_content_for_subsubtopic,
        current_subsubtopic_title,
        next_subsubtopic_title,
    )

    subsubtopic_info['subsubtopic_title'] = current_subsubtopic_title
    subsubtopic_info['subsubtopic_start_page_label'] = current_subsubtopic_page_label
    subsubtopic_info['subsubtopic_end_page_label'] = next_subsubtopic_page_label
    subsubtopic_info['subsubtopic_page_contents'] = subsubtopic_content

    return subsubtopic_info

In [15]:
import json
from datetime import datetime
import re

def save_chapter_content(chapter_content: dict, output_dir: str) -> str:
    """
    Save chapter content dictionary as a JSON file.
    
    Args:
        chapter_content: Dictionary containing chapter information with keys:
            - chapter_number: The chapter number
            - chapter_title: The chapter title
            - (other chapter data)
    
    Returns:
        The file path where the JSON was saved
    """
    # Get chapter number and title
    chapter_number = str(chapter_content.get('chapter_number', ''))
    chapter_title = chapter_content.get('chapter_title', '')
    
    # Sanitize chapter title for filename (remove invalid characters)
    # Replace spaces with underscores and remove special characters
    sanitized_title = re.sub(r'[^\w\s-]', '', chapter_title)
    sanitized_title = re.sub(r'[-\s]+', '_', sanitized_title)
    sanitized_title = sanitized_title.strip('_')
    
    # Get current date in YYYYMMDD format
    date_str = datetime.now().strftime('%Y%m%d')
    
    # Create filename: chapternumber_chapter_title_date_extracted.json
    filename = f"{chapter_number}_{sanitized_title}_{date_str}_extracted.json"
    
    # Create directory path
    output_dir = BASE_DIR_PATH / output_dir
    
    # Create directory if it doesn't exist
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Full file path
    file_path = output_dir / filename
    
    # Save as JSON with indentation for readability
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(chapter_content, f, indent=2, ensure_ascii=False)
    
    return str(file_path)

In [22]:
chapter_file_path = {}
output_dir = "docs/extracted_source_content/biology/essential-biology"
for chapter_idx in range(len(detailed_toc) - 1):
    chapter_content = {}
    current_chapter = detailed_toc[chapter_idx]
    next_chapter = detailed_toc[chapter_idx + 1]
    
    current_chapter_number = current_chapter['chapter_number']
    current_chapter_title = current_chapter['chapter_title']
    current_chapter_page_label = current_chapter['page_label']
    
    current_chapter_page_doc_index = page_label_map[current_chapter_page_label]
    current_chapter_page_contents = content_parser.parse_chapter_title_page(essential_biology_docs[current_chapter_page_doc_index].page_content, current_chapter_title)
    
    current_chapter_subtopics = current_chapter['subtopics']
    
    chapter_content['chapter_number'] = current_chapter_number
    chapter_content['chapter_title'] = current_chapter_title
    chapter_content['chapter_page_label'] = current_chapter_page_label
    chapter_content['outline'] = create_outline(current_chapter_subtopics)
    chapter_content['prerequisites'] = current_chapter_page_contents['prerequisites']
    chapter_content['introduction'] = current_chapter_page_contents['introduction']
    
    chapter_content['subtopics_info'] = []
    
    for subtopic_idx, subtopic in enumerate(current_chapter_subtopics):
        if subtopic_idx < len(current_chapter_subtopics) - 1:
            next_subtopic = current_chapter_subtopics[subtopic_idx + 1]
        else:
            next_subtopic = None

        subtopic_info = extract_subtopic_info(subtopic, next_subtopic, next_chapter)
        
        subtopic_info['subsubtopics_info'] = []
        
        for subsub_idx in range(len(subtopic['subsubtopics'])):
            current_subsubtopic = subtopic['subsubtopics'][subsub_idx]
            if subsub_idx < len(subtopic['subsubtopics']) - 1:
                next_subsubtopic = subtopic['subsubtopics'][subsub_idx + 1]
            else:
                next_subsubtopic = None
            
            subsubtopic_info = extract_subsubtopic_info(current_subsubtopic, next_subsubtopic, next_chapter)
            
            
            subtopic_info['subsubtopics_info'].append(subsubtopic_info) 
               
        chapter_content['subtopics_info'].append(subtopic_info)
    
    next_chapter_start_page_label = next_chapter['page_label']
    next_chapter_start_page_doc_index = page_label_map[next_chapter_start_page_label]
    
    chapter_figures_info = []
    for page_idx in range(current_chapter_page_doc_index, next_chapter_start_page_doc_index, 1):
        chapter_figures_info.extend(
            content_parser.extract_figure_detailed_content(
                essential_biology_docs[page_idx].page_content,
                current_chapter_number,
            )
        )
    
    chapter_content['figures_info'] = chapter_figures_info
    chapter_content['metadata'] = current_chapter_subtopics
    
    chapter_file_path[current_chapter_number] = save_chapter_content(chapter_content, output_dir)


In [28]:
len(detailed_toc)

32

In [22]:
chapter_file_path

{'1': 'c:\\Users\\Motunrayo Ibiyo\\Documents\\IBrary\\docs\\extracted_source_content\\biology\\essential-biology\\1_Biology_The_Science_of_Life_20260117_extracted.json',
 '2': 'c:\\Users\\Motunrayo Ibiyo\\Documents\\IBrary\\docs\\extracted_source_content\\biology\\essential-biology\\2_The_Chemical_Basis_of_Life_20260117_extracted.json'}

In [20]:
pprint.pp(essential_biology_docs[page_label_map['317']].page_content)

('CHAPTER 18  The Plants and Fungi 317\n'
 'leatherleaf fern\n'
 'hart’s\n'
 'tongue\n'
 'fern\n'
 'tree fern\n'
 'Figure 18.7 Diversity of ferns.\n'
 'All ferns are vascular plants that do not utilize seeds for reproduction.\n'
 '(leatherleaf fern): © Gregory Preest/Alamy; (tree fern): © Danita '
 'Delimont/Getty \n'
 'Images; (hart’s tongue fern): © Organics image library/Alamy RF \n'
 'sold as lycopodium powder, or vegetable sulfur, for use in pharma -\n'
 'ceuticals and in fireworks because it is highly flammable. The Lyco-\n'
 'podium featured in Figure 18.6\xa0 is common in moist woodlands in \n'
 'temperate climates, where they are called ground pines; they are also \n'
 'abundant in the tropics and subtropics.\n'
 'Ferns\n'
 'Ferns are a widespread group of plants that are well known for their \n'
 'attractiveness. Unlike lycophytes, ferns have megaphylls, or large \n'
 'leaves with branched veins. Megaphylls provide a large surface area for \n'
 'capturing the sunlight needed 

In [ ]:
page_label_map['311']

In [21]:
# Save TOC and Chapter 1 to JSON files
toc_output_path = BASE_DIR_PATH / "docs" / "textbooks" / "biology" / "essential_biology_toc.json"
toc_parser.save_toc_to_json({
    "brief_toc": brief_toc,
    "detailed_toc": detailed_toc,
    "page_label_map": page_label_map
}, toc_output_path)
print(f"TOC saved to: {toc_output_path}")

# Save Chapter 1
if chapter_1:
    chapter_1_output_path = BASE_DIR_PATH / "docs" / "textbooks" / "biology" / "essential_biology_chapter1.json"
    chapter_parser.save_to_json([chapter_1], chapter_1_output_path)
    print(f"Chapter 1 saved to: {chapter_1_output_path}")
else:
    print("No chapter data to save.")

TOC saved to: c:\Users\Motunrayo Ibiyo\Documents\IBrary\docs\textbooks\biology\essential_biology_toc.json


: 

In [ ]:
# Save Chapter 1 to JSON file
if chapter_1:
    output_path = BASE_DIR_PATH / "docs" / "textbooks" / "biology" / "essential_biology_chapter1.json"
    chapter_parser.save_to_json([chapter_1], output_path)
    print(f"Chapter 1 saved to: {output_path}")
else:
    print("No chapter data to save.")